In [11]:
import pandas as pd
import numpy as np
import datetime as dt
#load raw trasactional dataset
#why we have used encoding means "Western European languages, can decode any byte value"
df=pd.read_csv("data.csv",encoding="ISO-8859-1")
#to get datatype,range,total columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [12]:
#drop records missing a CustomerID
#removes all the rows which have NaN values
# creates a seperate copy of the dataframe
df_clean=df.dropna(subset=['CustomerID']).copy()
df_clean['CustomerID']=df_clean['CustomerID'].astype(int)
dropped_count = len(df) - len(df_clean)
print("Dropped rows count:", dropped_count)
df_clean.info()

Dropped rows count: 135080
<class 'pandas.core.frame.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    406829 non-null  object 
 1   StockCode    406829 non-null  object 
 2   Description  406829 non-null  object 
 3   Quantity     406829 non-null  int64  
 4   InvoiceDate  406829 non-null  object 
 5   UnitPrice    406829 non-null  float64
 6   CustomerID   406829 non-null  int64  
 7   Country      406829 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 27.9+ MB


In [13]:
df_returns=df_clean[(df_clean['Quantity']<=0)| (df_clean['UnitPrice']<0)]
df_returns.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8905 entries, 141 to 541717
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    8905 non-null   object 
 1   StockCode    8905 non-null   object 
 2   Description  8905 non-null   object 
 3   Quantity     8905 non-null   int64  
 4   InvoiceDate  8905 non-null   object 
 5   UnitPrice    8905 non-null   float64
 6   CustomerID   8905 non-null   int64  
 7   Country      8905 non-null   object 
dtypes: float64(1), int64(2), object(5)
memory usage: 626.1+ KB


In [14]:

df_sales=df_clean[(df_clean['Quantity']>0) & (df_clean['UnitPrice']>0)].copy()
df_sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 397884 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    397884 non-null  object 
 1   StockCode    397884 non-null  object 
 2   Description  397884 non-null  object 
 3   Quantity     397884 non-null  int64  
 4   InvoiceDate  397884 non-null  object 
 5   UnitPrice    397884 non-null  float64
 6   CustomerID   397884 non-null  int64  
 7   Country      397884 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 27.3+ MB


In [15]:
#calculate total revenue per line item
df_sales['Line_Revenue']=df_sales['Quantity']*df_sales['UnitPrice']
#it changes from string to datetime
#because invoice date is read as string 
df_sales['InvoiceDate']=pd.to_datetime(df_sales['InvoiceDate'])
df_sales['Year']=df_sales['InvoiceDate'].dt.year
df_sales['Month']=df_sales['InvoiceDate'].dt.month
df_sales['DayOfWeek']=df_sales['InvoiceDate'].dt.day_name()
df_sales['Hour']=df_sales['InvoiceDate'].dt.hour
#data loading and processing completed
# The CSV file has been successfully loaded.
# Missing CustomerID rows have been removed.
# Returns/cancellations have been separated from actual sales.
# Additional columns (such as Line_Revenue, Year, Month, etc.) have been created.
# The data is now ready for further analysis.
print(f"Raw records:{len(df)} | clean sales rows: {len(df_sales)} | Returns Isolated: {len(df_returns)}")

Raw records:541909 | clean sales rows: 397884 | Returns Isolated: 8905


In [16]:
#set evaluation snapshot date (1 day after the absolute maximum date in the dataset)
snapshot_date=df_sales['InvoiceDate'].max()+dt.timedelta(days=1)
#group transactions by unique ID
rfm=df_sales.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date-x.max()).days,
    'InvoiceNo':'nunique',
    'Line_Revenue':'sum'
}).reset_index()
#rename columns to clear business syntax
rfm.columns=['CustomerID','Recency','Frequency','Monetary']
rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


In [17]:
#rank assigning
import pandas as pd
import numpy as np
import datetime as dt

# 1. Safely isolate and calculate absolute return values per customer
df_returns = df_returns.copy()
df_returns['Return_Value'] = df_returns['Quantity'] * df_returns['UnitPrice']
customer_returns = df_returns.groupby('CustomerID')['Return_Value'].sum().abs().reset_index()

# 2. Reset baseline RFM to allow safe cell re-runs without column duplication
snapshot_date = df_sales['InvoiceDate'].max() + dt.timedelta(days=1)
rfm = df_sales.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'Line_Revenue': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# 3. Merge returns and surgically handle null spaces without touching categorical scales
rfm = rfm.merge(customer_returns, on='CustomerID', how='left')
rfm['Return_Value'] = rfm['Return_Value'].fillna(0)

# 4. Net revenue calculation accounting for merchandise returns
rfm['Net_Monetary'] = rfm['Monetary'] - rfm['Return_Value']

# 5. Rank and divide customers into quintiles using true Net Value
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Net_Monetary'], q=5, labels=[1, 2, 3, 4, 5])

# 6. Corporate Persona Mapping Function
def assign_segment(row):
    r, f, m = int(row['R_Score']), int(row['F_Score']), int(row['M_Score'])
    if r >= 4 and f >= 4 and m >= 4: return 'Champions'
    elif r >= 3 and f >= 3: return 'Loyal Customers'
    elif r <= 2 and (f >= 3 or m >= 3): return 'At Risk'
    else: return 'Hibernating / Lost'

rfm['Customer_Segment'] = rfm.apply(assign_segment, axis=1)
rfm.to_csv("rfm_output.csv", index=False)
print(" Clean rfm_output.csv successfully generated!")



 Clean rfm_output.csv successfully generated!


In [18]:
#EDA
print("===1. TOP 10 HIGHEST SELLING PRODUCTS===")
top_products= df_sales.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
print(top_products)

print("\n=== 2.MONTHLY REVENUE DISTRIBUTION===")
monthly_revenue=df_sales.groupby(['Year','Month'])['Line_Revenue'].sum().reset_index()
print(monthly_revenue)

print("\n===3. REAK TRAFFIC HOURS(TIME SERIES PATTERN)===")
hourly_traffic=df_sales.groupby('Hour')['InvoiceNo'].nunique()
print(hourly_traffic)

print("\n===REVENUE BY COUNTRY===")
country_revenue=df_sales.groupby('Country')['Line_Revenue'].sum().sort_values(ascending=False).head(5)
print(country_revenue)

===1. TOP 10 HIGHEST SELLING PRODUCTS===
Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        77916
WORLD WAR 2 GLIDERS ASSTD DESIGNS     54415
JUMBO BAG RED RETROSPOT               46181
WHITE HANGING HEART T-LIGHT HOLDER    36725
ASSORTED COLOUR BIRD ORNAMENT         35362
PACK OF 72 RETROSPOT CAKE CASES       33693
POPCORN HOLDER                        30931
RABBIT NIGHT LIGHT                    27202
MINI PAINT SET VINTAGE                26076
Name: Quantity, dtype: int64

=== 2.MONTHLY REVENUE DISTRIBUTION===
    Year  Month  Line_Revenue
0   2010     12    572713.890
1   2011      1    569445.040
2   2011      2    447137.350
3   2011      3    595500.760
4   2011      4    469200.361
5   2011      5    678594.560
6   2011      6    661213.690
7   2011      7    600091.011
8   2011      8    645343.900
9   2011      9    952838.382
10  2011     10   1039318.790
11  2011     11   1161817.380
12  2011     12    518192.790

===3. REAK TRAFFIC

In [19]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Group transactions into daily time-series buckets
df_sales['Date'] = df_sales['InvoiceDate'].dt.date
daily_sales = df_sales.groupby('Date')['Line_Revenue'].sum().reset_index()
daily_sales['Date'] = pd.to_datetime(daily_sales['Date'])
daily_sales.set_index('Date', inplace=True)
daily_sales = daily_sales.asfreq('D', fill_value=0)

# Train model with a 7-day seasonal period to eliminate convergence warnings
model = ExponentialSmoothing(daily_sales['Line_Revenue'], trend='add', seasonal='add', seasonal_periods=7)
fitted_model = model.fit()

# Generate and export 30-day forecast horizon
forecast_horizon = 30
forecast_values = fitted_model.forecast(steps=forecast_horizon)
future_dates = pd.date_range(start=daily_sales.index.max() + pd.Timedelta(days=1), periods=forecast_horizon)

df_forecast = pd.DataFrame({
    'Date': future_dates,
    'Forecasted_Revenue': np.round(forecast_values, 2)
}).reset_index(drop=True)

df_forecast.to_csv("sales_forecast_output.csv", index=False)
print("Optimized sales_forecast_output.csv successfully generated!")



Optimized sales_forecast_output.csv successfully generated!


c:\Users\Admin\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


In [20]:
# Save the daily historical baseline data to a clean CSV
daily_sales.to_csv("historical_daily_sales.csv", index=True)
print("Success! 'historical_daily_sales.csv' is generated.")

Success! 'historical_daily_sales.csv' is generated.
